# PS 3D:
Detect the faults in Sensor Data from Machinery (NASA Turbofan Engine Dataset)
using Autoencoder and build the model by using the following steps:
  
  • Dataset: The dataset contains sensor readings from turbofan engines with labels
indicating whether a reading is faulty.
  
  • Goal: Detect machinery faults by training an autoencoder on sensor data from healthy
engines.

Steps:
  
  • Preprocess data by scaling sensor readings.
  
  • Use an LSTM-based autoencoder (to capture sequential data) to learn patterns in
normal sensor readings.
  
  • Calculate reconstruction error for test data, with a threshold to classify high-error
instances as faults.

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow.keras import models
from tensorflow.keras import layers

In [7]:
# LOADING THE DATASET
data = pd.read_csv("NASA TurboFan Engine.csv")

data.head()
# print(data.shape)

,unit,cycle,setting1,setting2,setting3,sensor1,sensor2,sensor3,sensor4,sensor5,...,sensor14,sensor15,sensor16,sensor17,sensor18,sensor19,sensor20,sensor21,RUL,Fault_Label
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191,0
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190,0
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189,0
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188,0
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187,0


In [8]:
# PREPROCESSING AND SPLITTING X AND Y
data = data.fillna(data.mean(numeric_only = True))

X = data.drop("Fault_Label", axis=1).values
y = data["Fault_Label"].values


In [9]:
# TRAIN AND TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state = 42,
    test_size = 0.2
)

In [10]:
# KEEP ONLY NORMAL DATA FOR TRAINING
X_train = X_train[y_train == 0]

In [11]:
# NORMALIZE THE DATA
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [12]:
# SPLIT TRAIN DATA INTO TRAINING AND VALIDATION
X_train, x_val = train_test_split(
    X_train,
    test_size = 0.1, 
    random_state = 42
)

In [13]:
# RESHAPE DATA FOR LSTM
X_train = X_train.reshape(-1, X_train.shape[1], 1)
X_val = x_val.reshape(-1, x_val.shape[1], 1)
X_test = X_test.reshape(-1, X_test.shape[1],1)

In [15]:
# DEFINING AUTOENCODER ARCHITECTURE

# Encoder
encoder = models.Sequential([
    layers.Input(shape=(X_train.shape[1],1)),
    layers.LSTM(16)
])

# Decoder
decoder = models.Sequential([
    layers.Input(shape=(16,)),
    layers.RepeatVector(X_train.shape[1]),
    layers.LSTM(16, return_sequences=True),
    layers.TimeDistributed(layers.Dense(1))
])

# Autoencoder
model = models.Sequential([encoder, decoder])

In [16]:
# COMPILE AND TRAIN THE MODEL
model.compile(
    optimizer = 'adam',
    loss = 'mse',
    metrics = ['mae']
)

history = model.fit(
    X_train,X_train,
    validation_data=(X_val, X_val),
    epochs = 11,
    batch_size = 256
)

Epoch 1/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 8s 51ms/step - loss: 0.7224 - mae: 0.5900 - val_loss: 0.6935 - val_mae: 0.6024
Epoch 2/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.6740 - mae: 0.5974 - val_loss: 0.6486 - val_mae: 0.5894
Epoch 3/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 0.6466 - mae: 0.5855 - val_loss: 0.6382 - val_mae: 0.5820
Epoch 4/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 31ms/step - loss: 0.6386 - mae: 0.5789 - val_loss: 0.6303 - val_mae: 0.5744
Epoch 5/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 0.6375 - mae: 0.5775 - val_loss: 0.6326 - val_mae: 0.5773
Epoch 6/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - loss: 0.6329 - mae: 0.5739 - val_loss: 0.6257 - val_mae: 0.5707
Epoch 7/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - loss: 0.6295 - mae: 0.5709 - val_loss: 0.6246 - val_mae: 0.5695
Epoch 8/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 30ms/step - loss: 0.6298 - mae: 0.5716 - val_loss: 0.6219 - val_mae: 0.5684
Epoch 9/11
50/50 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - loss: 0.625

In [17]:
# EVALUATE THE MODEL
test_loss, test_accuracy = model.evaluate(X_test, X_test)

print("Test loss : ", test_loss)
print("Test accuracy: " , test_accuracy)


129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.9132 - mae: 0.6571
Test loss :  0.9132413268089294
Test accuracy:  0.6571179628372192


In [20]:
# CALCULATING RECONSTRCUTION ERROR AND SETTING THRESHOLD
train_pred = model.predict(X_val)

train_error = np.mean(
    np.square(X_test - test_pred) , axis = (1, 2)
)

threshold = np.percentile(train_error, 95)

print("Threshold: ", threshold)


44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
Threshold:  2.9679181510895956


In [21]:
# DETECT ANOMALIES
test_pred = model.predict(X_test)

test_error = np.mean(
    np.square(X_test - test_pred), axis=(1,2)
)

predictions = ( test_error > threshold ).astype(int)

print("Acutal: ", y_test[:10])
print("Predicted: " , predictions[:10])

129/129 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
Acutal:  [0 1 0 1 0 0 0 0 0 0]
Predicted:  [0 1 0 1 0 0 0 0 0 0]


In [24]:
# EVALUATE THE MODEL
print("Confusion Matrix: ")
print(confusion_matrix(y_test, predictions))

print("Classification Report" )
print(classification_report(y_test, predictions))

Confusion Matrix: 
[[3541    2]
 [ 379  205]]
Classification Report
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      3543
           1       0.99      0.35      0.52       584

    accuracy                           0.91      4127
   macro avg       0.95      0.68      0.73      4127
weighted avg       0.92      0.91      0.89      4127

